# Ex-vivo Instrument Spread Analysis

Companion to [`surgical_force_analysis.ipynb`](surgical_force_analysis.ipynb) for a
**different data layout**. Here every trial for every participant lives in a **single
tabular file**; each row carries one metric for one instrument, and the actual samples
are packed as a comma-separated string in the `data` column.

| column | meaning |
|---|---|
| `participant` | participant id |
| `trial` | trial label (e.g. `Trial1`) |
| `expertise` | expertise group (e.g. `Senior`) |
| `level` | training level (e.g. `Fellow/Spine`) |
| `instrument` | `scissors`, `cavitron` or `bipolar` |
| `metric` | `position`, `timestamp`, `captured_flag`, `velocity`, … |
| `data` | comma-separated samples (position is `x,y,z,x,y,z,…`) |
| `len` | number of frames in the trial |

This notebook reproduces the **spread / instrument-use localization** analysis from
§11b of the surgical notebook — the covariance-ellipsoid volume, its anisotropy, and the
dwell-time-weighted occupancy — for the **`position`** of **scissors, cavitron and
bipolar** only (velocity, acceleration, … are ignored). It reports the spread three ways:
**per participant**, **per expertise**, and **per level**, with figures.

Assumptions, per the data spec:
- `timestamp` is shared by all rows of the same `participant`+`trial`.
- Each instrument's `position` is filtered by that instrument's own `captured_flag`
  (only frames where the flag is true are kept).


## 1 · Setup

In [ ]:
# Colab already ships these; this is a no-op there and a convenience elsewhere.
import importlib, subprocess, sys
for pkg in ("numpy", "scipy", "matplotlib", "pandas"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="All-NaN slice encountered")

# ---- Plot style (matches the surgical notebook) -------------------------
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.labelsize": 10, "legend.fontsize": 9, "legend.frameon": False,
    "font.size": 10,
})

# Analyse position for these three instruments only.
INSTRUMENTS = ["Bipolar", "Cavitron", "Scissors"]
# Okabe-Ito colorblind-safe palette, fixed order per instrument.
INST_COLOR = {"Bipolar": "#0072B2", "Cavitron": "#E69F00", "Scissors": "#009E73"}
# Canonical (Title-case) names <- raw lowercase names found in the file.
INST_CANON = {"bipolar": "Bipolar", "cavitron": "Cavitron", "scissors": "Scissors"}

VOXEL_SIZE = 2.0        # mm; voxel edge for the occupancy-entropy (spatial concentration) metric

# Report figures / CSVs are written here (created if missing) so it works anywhere,
# including Colab where there is no pre-existing data/ folder.
OUTPUT_DIR = "exvivo_report"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Setup complete. Report artifacts -> {OUTPUT_DIR}/")

## 2 · Load the data from Google Drive

On Colab the drive is mounted and `DATA_FILE` points at the `full_data.json` export. The
loader also accepts `.csv`/`.tsv` (delimiter auto-sniffed; a comma `.csv` needs the `data`
column quoted since it contains commas), `.parquet`, `.pickle` and `.xlsx`. Off Colab,
just set `DATA_FILE` to a local path.

In [ ]:
# Mount Google Drive when running on Colab (no-op elsewhere).
try:
    from google.colab import drive
    drive.mount("/content/drive")
except (ImportError, ModuleNotFoundError):
    pass

DATA_FILE = "/content/drive/MyDrive/Colab Notebooks/Analyze Surgical Data/full_data.json"

def load_table(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".json":
        return pd.read_json(path)
    if ext in (".parquet", ".pq"):
        return pd.read_parquet(path)
    if ext in (".pkl", ".pickle"):
        return pd.read_pickle(path)
    if ext in (".xlsx", ".xls"):
        return pd.read_excel(path)
    # csv / tsv / txt: let pandas sniff the delimiter (handles comma-in-quotes and tabs)
    return pd.read_csv(path, sep=None, engine="python")

df = load_table(DATA_FILE)
df.columns = [c.strip().lower() for c in df.columns]
df["instrument"] = df["instrument"].astype(str).str.strip().str.lower()
df["metric"] = df["metric"].astype(str).str.strip().str.lower()
df["participant"] = df["participant"].astype(str).str.strip()

print(f"{len(df)} rows | "
      f"{df['participant'].nunique()} participants | "
      f"metrics: {sorted(df['metric'].unique())}")
df.head()

## 3 · Parse each instrument's captured position

For every `participant`+`trial` we read the shared `timestamp`, then for each instrument
pull its `position` (reshaped to `x,y,z` per frame) and its own `captured_flag`, keeping
only the frames where the flag is true. `dt` (per-frame dwell time) comes from the
timestamp and is used later to time-weight the occupancy metric.

In [ ]:
_TRUE = ("1", "true", "t", "yes")

def _floats(s):
    "data cell -> float array; accepts a comma-separated string or an actual list/array."
    if isinstance(s, (list, tuple, np.ndarray)):
        return np.asarray(s, dtype=float)
    return np.fromstring(str(s).strip("[]"), sep=",", dtype=float)

def _bools(s):
    "data cell -> bool array; accepts a comma-separated string or a list of bools/strings."
    if isinstance(s, (list, tuple, np.ndarray)):
        return np.array([v if isinstance(v, (bool, np.bool_))
                         else str(v).strip().lower() in _TRUE for v in s], bool)
    return np.array([x.strip().lower() in _TRUE
                     for x in str(s).split(",")], bool)

def _first(g, metric, instrument=None):
    "First matching row's data cell, or None."
    m = g["metric"] == metric
    if instrument is not None:
        m &= g["instrument"] == instrument
    rows = g[m]
    return rows.iloc[0]["data"] if len(rows) else None

trials = []          # one record per (participant, trial, instrument) with captured position
for (pid, trial), g in df.groupby(["participant", "trial"], sort=False):
    exp = g["expertise"].iloc[0]
    lvl = g["level"].iloc[0]
    ts = _first(g, "timestamp")                    # shared across the trial
    t = _floats(ts) if ts is not None else None
    for inst_l, inst in INST_CANON.items():
        pos = _first(g, "position", inst_l)
        cap = _first(g, "captured_flag", inst_l)
        if pos is None or cap is None:
            continue
        P = _floats(pos).reshape(-1, 3)            # (N, 3) tip positions
        c = _bools(cap)                            # (N,) per-instrument validity
        n = min(len(P), len(c))                    # guard against off-by-one lengths
        P, c = P[:n], c[:n]
        dt = np.gradient(t[:n]) if (t is not None and len(t) >= n) else np.ones(n)
        trials.append(dict(
            participant=str(pid), trial=str(trial), expertise=exp, level=lvl,
            instrument=inst,
            pos=P[c],                              # captured positions only
            dt=dt[c],                              # matching dwell weights
            n_total=int(n), n_captured=int(c.sum()),
        ))

print(f"{len(trials)} (participant x trial x instrument) records parsed")
print("participants:", sorted({r['participant'] for r in trials}))
print("expertise groups:", sorted({r['expertise'] for r in trials}))
print("levels:", sorted({r['level'] for r in trials}))

## 4 · Spread / instrument-use localization metrics

Identical to §11b of the surgical notebook, computed over each instrument's **captured**
`position` samples:

- **Covariance-ellipsoid volume** `(4/3)π·√(λ₁λ₂λ₃)` — orientation-invariant overall
  spread. The eigenvalues `λ` of the 3×3 position covariance are variances along the
  cloud's own principal axes; the 1-σ ellipsoid has semi-axes `√λ`.
- **Anisotropy** `λ₁/λ₃` — motion confined to a line/plane (large) vs. an isotropic blob (≈1).
- **Occupancy `exp(H)`** — dwell-time-weighted effective number of occupied
  `VOXEL_SIZE` (2 mm) voxels; low means the tip keeps revisiting a tight core.

Smaller ellipsoid volume / occupancy ⇒ more localized use.

In [ ]:
def spread_metrics(P, w):
    "P: (M,3) captured positions; w: (M,) dwell weights. Returns the three spread metrics."
    if P.shape[0] < 2:
        return dict(ell_vol=np.nan, aniso=np.nan, occ_eff=np.nan, occ_voxels=0)
    # covariance-ellipsoid volume + anisotropy (frame-independent)
    cov = np.cov(P, rowvar=False)
    lam = np.clip(np.sort(np.linalg.eigvalsh(cov))[::-1], 0.0, None)      # lam1>=lam2>=lam3
    ell = float((4.0 / 3.0) * np.pi * np.sqrt(np.prod(lam)))              # mm^3
    aniso = float(lam[0] / lam[2]) if lam[2] > 0 else np.nan
    # dwell-time-weighted occupancy entropy -> effective #occupied voxels exp(H)
    keys = np.floor(P / VOXEL_SIZE).astype(np.int64)
    _, inv = np.unique(keys, axis=0, return_inverse=True)
    inv = np.asarray(inv).ravel()
    wsum = np.zeros(inv.max() + 1)
    np.add.at(wsum, inv, w)
    wsum = wsum[wsum > 0]
    if wsum.sum() > 0:
        p = wsum / wsum.sum()
        H = float(-np.sum(p * np.log(p)))
        occ_eff, occ_vox = float(np.exp(H)), int(p.size)
    else:
        occ_eff, occ_vox = np.nan, 0
    return dict(ell_vol=ell, aniso=aniso, occ_eff=occ_eff, occ_voxels=occ_vox)

rows = []
for r in trials:
    m = spread_metrics(r["pos"], r["dt"])
    rows.append({**{k: r[k] for k in
                    ("participant", "trial", "expertise", "level",
                     "instrument", "n_total", "n_captured")}, **m})

R = pd.DataFrame(rows)
# keep a stable instrument order for plotting/tables
R["instrument"] = pd.Categorical(R["instrument"], categories=INSTRUMENTS, ordered=True)
R = R.sort_values(["participant", "trial", "instrument"]).reset_index(drop=True)
R

## 5 · Individual spread per participant

The three localization panels, one group of bars per participant, colored by instrument —
the direct analogue of §11b. Error bars are the SD across that participant's trials.

In [ ]:
PANELS = [
    ("ell_vol", "ellipsoid volume", "mm\u00b3"),
    ("aniso",   "anisotropy \u03bb\u2081/\u03bb\u2083", "ratio"),
    ("occ_eff", "occupancy exp(H)", "eff. voxels"),
]

def _grouped_bar(ax, groups, series, ylabel, title, xlabel, colors):
    "series: dict label -> (means, errs); one group of bars per entry in `groups`."
    x = np.arange(len(groups))
    k = len(series)
    w = 0.8 / k
    for i, (lab, (means, errs)) in enumerate(series.items()):
        means, errs = np.asarray(means, float), np.asarray(errs, float)
        yerr = np.vstack([np.minimum(errs, means), errs])
        off = (i - (k - 1) / 2) * w
        ax.bar(x + off, means, width=w * 0.95, yerr=yerr, label=lab, color=colors[i],
               error_kw=dict(lw=1, capsize=3, ecolor="#555"))
    ax.set_xticks(x)
    ax.set_xticklabels(groups, rotation=30 if max(len(str(g)) for g in groups) > 6 else 0,
                       ha="right" if max(len(str(g)) for g in groups) > 6 else "center")
    ax.set_ylabel(ylabel); ax.set_title(title); ax.set_xlabel(xlabel)
    ax.margins(y=0.15)
    if k > 1:
        ax.legend(fontsize=8)

def spread_figure(group_col, order=None, suptitle=None):
    "One 3-panel figure of the spread metrics, grouped by `group_col`, bars per instrument."
    groups = list(order) if order is not None else sorted(R[group_col].dropna().unique())
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
    for ax, (col, name, unit) in zip(axes, PANELS):
        series = {}
        for inst in INSTRUMENTS:
            means, errs = [], []
            for grp in groups:
                v = R[(R[group_col] == grp) & (R["instrument"] == inst)][col]
                v = v[np.isfinite(v)]
                means.append(v.mean() if len(v) else 0.0)
                errs.append(v.std() if len(v) > 1 else 0.0)
            series[inst] = (means, errs)
        _grouped_bar(ax, groups, series, unit, name, group_col,
                     [INST_COLOR[i] for i in INSTRUMENTS])
    fig.suptitle(suptitle or f"Instrument-use spread per {group_col}",
                 fontweight="bold", fontsize=13)
    fig.tight_layout()
    return fig

pids = sorted(R["participant"].unique())
fig = spread_figure("participant", order=pids,
                    suptitle="Instrument-use spread per participant")
fig.savefig(os.path.join(OUTPUT_DIR, "exvivo_spread_per_participant.png"), bbox_inches="tight")
plt.show()

## 6 · Overall view per expertise

Every participant/trial pooled by `expertise`; bars are the mean across all trials in the
group, error bars the SD.

In [ ]:
exp_order = sorted(R["expertise"].dropna().unique())
fig = spread_figure("expertise", order=exp_order,
                    suptitle="Instrument-use spread per expertise")
fig.savefig(os.path.join(OUTPUT_DIR, "exvivo_spread_per_expertise.png"), bbox_inches="tight")
plt.show()

## 7 · Overall view per level

In [ ]:
lvl_order = sorted(R["level"].dropna().unique())
fig = spread_figure("level", order=lvl_order,
                    suptitle="Instrument-use spread per level")
fig.savefig(os.path.join(OUTPUT_DIR, "exvivo_spread_per_level.png"), bbox_inches="tight")
plt.show()

## 8 · Tables and export

Per-trial spread for every instrument, plus group summaries (mean ± SD) by participant,
expertise and level. All are written to CSV alongside the data.

In [ ]:
def _summary(group_col):
    g = (R.groupby([group_col, "instrument"], observed=True)[["ell_vol", "aniso", "occ_eff"]]
           .agg(["mean", "std", "count"]))
    g.columns = [f"{m}_{s}" for m, s in g.columns]
    return g.reset_index()

per_trial = R.round({"ell_vol": 1, "aniso": 2, "occ_eff": 1})
per_trial.to_csv(os.path.join(OUTPUT_DIR, "exvivo_spread_per_trial.csv"), index=False)

for col in ("participant", "expertise", "level"):
    _summary(col).round(2).to_csv(
        os.path.join(OUTPUT_DIR, f"exvivo_spread_by_{col}.csv"), index=False)

print("Wrote to", OUTPUT_DIR + "/:")
for fn in ("exvivo_spread_per_trial.csv", "exvivo_spread_by_participant.csv",
           "exvivo_spread_by_expertise.csv", "exvivo_spread_by_level.csv",
           "exvivo_spread_per_participant.png", "exvivo_spread_per_expertise.png",
           "exvivo_spread_per_level.png"):
    print("  " + fn)

print("\nSpread by expertise (mean):")
display(_summary("expertise").round(1))
print("Spread by level (mean):")
display(_summary("level").round(1))
per_trial

---
All figures (`exvivo_spread_per_participant.png`, `…_per_expertise.png`,
`…_per_level.png`) and tables (`exvivo_spread_per_trial.csv` plus the grouped summaries)
are written to the `OUTPUT_DIR` folder (`exvivo_report/` by default). On Colab, point
`OUTPUT_DIR` at a Drive path if you want them saved back to Google Drive.